# Lab: Different Ways to Chunk a Podcast and a PDF
**Author:** Ayhan Demir · Ironhack AI Consulting & Integration Bootcamp · Module 3

**Goal:** chunk two very different sources with several strategies, measure the chunks, and prove one retrieval answer is grounded in inspectable source text.

| Source | File | Shape of the text |
|---|---|---|
| Podcast | `The_Blueprint_For_Trustworthy_AI.m4a` → `transcript.txt` | One long block of spoken text, no line breaks |
| PDF | `ai_hleg_ethics_guidelines_for_trustworthy_ai-en.pdf` | 41 pages, headings, numbered sections, many line breaks |

**Files this notebook creates:** `transcript.txt`, `chunk_stats.csv`, `clean_boundaries.csv`, `fig_size_distributions.png`, `fig_clean_boundaries.png`, `fig_semantic_similarity.png`, `lab_proof.md`

## Step 1: Setup and Data Loading
### 1.1 Install packages
Transcription also needs **ffmpeg** installed on the computer (`ffmpeg -version` in a terminal to check).

In [1]:
print("hello")

hello


In [2]:
%pip install langchain-text-splitters pypdf openai tiktoken sentence-transformers matplotlib pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


### 1.2 Imports
Expected output: no errors, the three splitter classes import.

In [1]:
import os, re, subprocess, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pypdf import PdfReader

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
print("Imports OK")

Imports OK


### 1.3 Load the API key from `.env`
The key is only needed for transcription. It is never printed or written into the notebook.

In [2]:
load_dotenv()  # reads .env in the current folder
print("API key found" if os.getenv("OPENAI_API_KEY") else "No API key found, check your .env file")

API key found


### 1.4 Load the podcast audio file

In [3]:
AUDIO_PATH = Path("The_Blueprint_For_Trustworthy_AI.m4a")
print("Current working folder:", Path.cwd())
print(f"Audio found: {AUDIO_PATH.name} ({AUDIO_PATH.stat().st_size / 1_000_000:.1f} MB)" if AUDIO_PATH.exists()
      else "Audio NOT FOUND, move the file into the folder shown above")

Current working folder: /Users/macbook-air/Desktop/AI consulting/Week3/Day2/lab-chunking-strategies
Audio found: The_Blueprint_For_Trustworthy_AI.m4a (0.6 MB)


### 1.5 Transcribe the audio and save it to `transcript.txt`
The API accepts files up to 25 MB, so ffmpeg first shrinks the audio (mono, 64 kbps mp3) and cuts it into 10-minute pieces. Whisper transcribes each piece, and the pieces are joined in order. If `transcript.txt` already exists, the function skips all of this, so re-running the notebook costs nothing.

In [15]:
import subprocess, glob

TRANSCRIPT_PATH = Path("transcript.txt")

def transcribe_audio(audio_path, out_path, model="whisper-1", piece_seconds=600):
    """Transcribe audio with the OpenAI API and save the text. Skips all work if out_path already exists."""
    out = Path(out_path)
    if out.exists() and out.stat().st_size > 0:
        print(f"{out} already exists -> skipping transcription")
        return out.read_text(encoding="utf-8")

    # 1) shrink and cut the audio: mono, 64 kbps mp3, 10-minute pieces
    Path("_pieces").mkdir(exist_ok=True)
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", str(audio_path), "-ac", "1", "-b:a", "64k",
                    "-f", "segment", "-segment_time", str(piece_seconds), "-reset_timestamps", "1",
                    "_pieces/piece_%02d.mp3"], check=True)

    # 2) transcribe the pieces in order
    from openai import OpenAI
    client = OpenAI()                                  # uses OPENAI_API_KEY from the environment
    texts = []
    for f in sorted(glob.glob("_pieces/piece_*.mp3")):
        with open(f, "rb") as fh:
            texts.append(client.audio.transcriptions.create(model=model, file=fh).text)
        print(f, "done")

    text = " ".join(texts)
    out.write_text(text, encoding="utf-8")             # save to a local file
    return text

podcast_text = transcribe_audio(AUDIO_PATH, TRANSCRIPT_PATH)
print(f"Podcast transcript length: {len(podcast_text)} chars")
print(podcast_text[:200])


transcript.txt already exists -> skipping transcription
Podcast transcript length: 327 chars
So imagine for a second you're driving across I don't know a massive suspension bridge. OK. You don't pull over halfway across get out and demand to see the blueprints right. You don't interview the w


In [14]:
PDF_PATH = Path("'/Users/macbook-air/Desktop/AI consulting/Week3/Day2/lab-chunking-strategies/ai_hleg_ethics_guidelines_for_trustworthy_ai-en.pdf'")

def load_pdf(path=PDF_PATH):
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)

pdf_text = load_pdf()
print(f"PDF length: {len(pdf_text)} chars")
print(pdf_text[:200])


FileNotFoundError: [Errno 2] No such file or directory: "'/Users/macbook-air/Desktop/AI consulting/Week3/Day2/lab-chunking-strategies/ai_hleg_ethics_guidelines_for_trustworthy_ai-en.pdf'"